 # AAL Sales Analysis - Q4 2020
This notebook analyzes the Q4 2020 sales data for AAL (Australian Apparel Limited) across different states, demographic groups, and time periods.

In [64]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [65]:
# Data Loading
df = pd.read_csv('AusApparalSales4thQrt2020.csv')
df.head()

,Date,Time,State,Group,Unit,Sales
0,1-Oct-2020,Morning,WA,Kids,8,20000
1,1-Oct-2020,Morning,WA,Men,8,20000
2,1-Oct-2020,Morning,WA,Women,4,10000
3,1-Oct-2020,Morning,WA,Seniors,15,37500
4,1-Oct-2020,Afternoon,WA,Kids,3,7500


## 1. Data Wrangling

### 1a. Data Inspection

In [66]:
print(f"DataFrame shape: {df.shape}")
print("\nColumn types:")
print(df.dtypes)
print("\nInfo:")
print(df.info())


DataFrame shape: (7560, 6)

Column types:
Date       str
Time       str
State      str
Group      str
Unit     int64
Sales    int64
dtype: object

Info:
<class 'pandas.DataFrame'>
RangeIndex: 7560 entries, 0 to 7559
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Date    7560 non-null   str  
 1   Time    7560 non-null   str  
 2   State   7560 non-null   str  
 3   Group   7560 non-null   str  
 4   Unit    7560 non-null   int64
 5   Sales   7560 non-null   int64
dtypes: int64(2), str(4)
memory usage: 354.5 KB
None


### 1a: Data Cleaning

In [67]:
# Remove whitespaces from column names
df.columns = df.columns.str.strip()

# Remove whitespaces from string columns
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip()

print("Columns:", df.columns.tolist())
print("\nSample data:")
print(df.head())

Columns: ['Date', 'Time', 'State', 'Group', 'Unit', 'Sales']

Sample data:
         Date       Time State    Group  Unit  Sales
0  1-Oct-2020    Morning    WA     Kids     8  20000
1  1-Oct-2020    Morning    WA      Men     8  20000
2  1-Oct-2020    Morning    WA    Women     4  10000
3  1-Oct-2020    Morning    WA  Seniors    15  37500
4  1-Oct-2020  Afternoon    WA     Kids     3   7500


C:\Users\kpilgulwar\AppData\Local\Temp\ipykernel_10020\2125280494.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


In [68]:
# Check for missing values
print("\nMissing values (isna):")
print(df.isna().sum())
print("\nMissing values (notna):")
print(df.notna().sum())


Missing values (isna):
Date     0
Time     0
State    0
Group    0
Unit     0
Sales    0
dtype: int64

Missing values (notna):
Date     7560
Time     7560
State    7560
Group    7560
Unit     7560
Sales    7560
dtype: int64


### 1b: Recommendation for Handling Missing Data

 **Observation:** The dataset has **no missing values** in any column — all 7560 records are complete

 **Recommendation:**
 For this dataset, since there are no missing values, no treatment is needed. However, `dropna()` is still included in the code as a safety step in case the data changes in the future.

In [69]:
# drop rows with missing values
df = df.dropna()

In [70]:
# Dealing with duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()

# Data Cleaning
df['Date'] = pd.to_datetime(df['Date'], format='%d-%b-%Y')



Duplicate rows: 0


### 1c:  Data Normalization

  **Why Normalization over Standardization?**
  - **Normalization** (Min-Max scaling) scales data to a 0–1 range. Best when data does not follow a normal distribution.
  - **Standardization** (Z-score) centers data around mean=0, std=1. Best for normally distributed data.

Since sales data is often skewed (a few states sell much more), **normalization** is preferred here — it preserves the relative differences and is easier to interpret (0 = lowest, 1 = highest).

In [71]:
# min max normalization on 'Sales' column
df['Sales_normalized'] = (df['Sales'] - df['Sales'].min()) / (df['Sales'].max() - df['Sales'].min())

# print sales and normalized sales
print("\nSales and Normalized Sales:")
print(df[['Sales', 'Sales_normalized']].head())


Sales and Normalized Sales:
   Sales  Sales_normalized
0  20000          0.095238
1  20000          0.095238
2  10000          0.031746
3  37500          0.206349
4   7500          0.015873


### 1d: Aggregating Data
GroupBy Insights

  **GroupBy** splits data into groups based on a column, applies a function (sum, mean, etc.), and combines the results. This is
  essential for:
  - **Data chunking:** Breaking sales data by State or Group to analyze each segment independently.
  - **Data merging/aggregation:** Summarizing total sales per state, average units per group, etc.

  **Recommendation:** Use GroupBy for **aggregation** in this analysis — it lets us compare total and average sales across states
  and demographic groups, which directly answers the CEO's questions about revenue by state.

In [72]:
print("State wise total sales:")
state_sales = df.groupby('State')[['Sales', 'Unit']].sum().sort_values(by='Sales', ascending=False)
print(state_sales)

State wise total sales:
           Sales   Unit
State                  
VIC    105565000  42226
NSW     74970000  29988
SA      58857500  23543
QLD     33417500  13367
TAS     22760000   9104
NT      22580000   9032
WA      22152500   8861


In [73]:
# Group wise total sales
print("\nGroup wise total sales:")
group_sales = df.groupby('Group')[['Sales', 'Unit']].sum().sort_values(by='Sales', ascending=False)
print(group_sales)


Group wise total sales:
            Sales   Unit
Group                   
Men      85750000  34300
Women    85442500  34177
Kids     85072500  34029
Seniors  84037500  33615


In [74]:
print("=== State-wise Total Sales ===")
state_sales = df.groupby('State')[['Unit', 'Sales', 'Sales_normalized']].sum().sort_values('Sales', ascending=False)
print(state_sales)

print("\n=== Group-wise Total Sales ===")
group_sales = df.groupby('Group')[['Unit', 'Sales', 'Sales_normalized']].sum().sort_values('Sales', ascending=False)
print(group_sales)

print("\n=== State & Group wise Average Sales ===")
state_group_sales = df.groupby(['State', 'Group'])[['Unit', 'Sales', 'Sales_normalized']].mean()
print(state_group_sales)

=== State-wise Total Sales ===
        Unit      Sales  Sales_normalized
State                                    
VIC    42226  105565000        635.968254
NSW    29988   74970000        441.714286
SA     23543   58857500        339.412698
QLD    13367   33417500        177.888889
TAS     9104   22760000        110.222222
NT      9032   22580000        109.079365
WA      8861   22152500        106.365079

=== Group-wise Total Sales ===
          Unit     Sales  Sales_normalized
Group                                     
Men      34300  85750000        484.444444
Women    34177  85442500        482.492063
Kids     34029  85072500        480.142857
Seniors  33615  84037500        473.571429

=== State & Group wise Average Sales ===
                    Unit         Sales  Sales_normalized
State Group                                             
NSW   Kids     27.537037  68842.592593          0.405350
      Men      28.181481  70453.703704          0.415579
      Seniors  26.944444  67361

## 2. Data Analysis
### 2a. Descriptive Statistical Analysis

In [75]:
# descriptive statistical analysis on 'Sales' column
print("\n=== Descriptive Statistics for Sales ===")
# mean, median, mode, variance, standard deviation
print("Mean:", df['Sales'].mean())
print("Median:", df['Sales'].median())
print("Mode:", df['Sales'].mode()[0])
print("Variance:", df['Sales'].var())
print("Standard Deviation:", df['Sales'].std())
print("Min:", df['Sales'].min())
print("Max:", df['Sales'].max())

# descriptive statistical analysis on 'Unit' column
print("\n=== Descriptive Statistics for Unit ===")
# mean, median, mode, variance, standard deviation
print("Mean:", df['Unit'].mean())
print("Median:", df['Unit'].median())
print("Mode:", df['Unit'].mode()[0])
print("Variance:", df['Unit'].var())
print("Standard Deviation:", df['Unit'].std())
print("Min:", df['Unit'].min())
print("Max:", df['Unit'].max())

# overall summary statistics
print("\n=== Overall Summary Statistics ===")
df[['Sales', 'Unit']].describe()


=== Descriptive Statistics for Sales ===
Mean: 45013.5582010582
Median: 35000.0
Mode: 22500
Variance: 1040288710.1844678
Standard Deviation: 32253.506943966073
Min: 5000
Max: 162500

=== Descriptive Statistics for Unit ===
Mean: 18.00542328042328
Median: 14.0
Mode: 9
Variance: 166.44619362951485
Standard Deviation: 12.90140277758643
Min: 2
Max: 65

=== Overall Summary Statistics ===


,Sales,Unit
count,7560.000000,7560.000000
mean,45013.558201,18.005423
std,32253.506944,12.901403
min,5000.000000,2.000000
25%,20000.000000,8.000000
50%,35000.000000,14.000000
75%,65000.000000,26.000000
max,162500.000000,65.000000


### 2b & 2c: Group with highest and lowest sales

In [76]:
# 2b & 2c: Group with highest and lowest sales
group_sales = df.groupby('Group')[['Unit','Sales']].sum().sort_values(by='Sales', ascending=False)
print("\nGroup-wise Total Sales:")
print(group_sales)
highest_sales_group = group_sales.idxmax()
lowest_sales_group = group_sales.idxmin()
print(f"\nGroup with highest sales:\n{highest_sales_group}")
print(f"\nGroup with lowest sales:\n{lowest_sales_group}")

# State with highest and lowest sales
state_sales = df.groupby('State')[['Unit','Sales']].sum().sort_values(by='Sales', ascending=False)
print("\nState-wise Total Sales:")
print(state_sales)
highest_sales_state = state_sales.idxmax()
lowest_sales_state = state_sales.idxmin()
print(f"\nState with highest sales:\n{highest_sales_state}")
print(f"\nState with lowest sales:\n{lowest_sales_state}")


Group-wise Total Sales:
          Unit     Sales
Group                   
Men      34300  85750000
Women    34177  85442500
Kids     34029  85072500
Seniors  33615  84037500

Group with highest sales:
Unit     Men
Sales    Men
dtype: str

Group with lowest sales:
Unit     Seniors
Sales    Seniors
dtype: str

State-wise Total Sales:
        Unit      Sales
State                  
VIC    42226  105565000
NSW    29988   74970000
SA     23543   58857500
QLD    13367   33417500
TAS     9104   22760000
NT      9032   22580000
WA      8861   22152500

State with highest sales:
Unit     VIC
Sales    VIC
dtype: str

State with lowest sales:
Unit     WA
Sales    WA
dtype: str


In [77]:
# print to see df again
print("\nDataFrame after cleaning and analysis:")
print(df.head())


DataFrame after cleaning and analysis:
        Date       Time State    Group  Unit  Sales  Sales_normalized
0 2020-10-01    Morning    WA     Kids     8  20000          0.095238
1 2020-10-01    Morning    WA      Men     8  20000          0.095238
2 2020-10-01    Morning    WA    Women     4  10000          0.031746
3 2020-10-01    Morning    WA  Seniors    15  37500          0.206349
4 2020-10-01  Afternoon    WA     Kids     3   7500          0.015873


### 2d: Weekly, monthly and Quarterly report

In [ ]:
# Set Date as index for time-based grouping
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
df['Month'] = df['Date'].dt.month_name()
df['Month_num'] = df['Date'].dt.month

# weekly report
print("\n=== Weekly Sales Report ===")
weekly_sales = df.groupby('Week')[['Unit', 'Sales']].sum()
print(weekly_sales)


=== Weekly Sales Report ===
       Unit     Sales
Week                 
40     6018  15045000
41    10801  27002500
42    10656  26640000
43    10726  26815000
44     8723  21807500
45     8346  20865000
46     8469  21172500
47     8445  21112500
48     8591  21477500
49    11849  29622500
50    12610  31525000
51    12662  31655000
52    12708  31770000
53     5517  13792500


In [80]:
# monthly report
print("\n=== Monthly Sales Report ===")
monthly_sales = df.groupby([('Month_num'), ('Month')])[['Unit', 'Sales']].sum()
# remove month_num index and replace with month names
monthly_sales.index = monthly_sales.index.droplevel(0)
print(monthly_sales)


=== Monthly Sales Report ===
           Unit      Sales
Month                     
October   45716  114290000
November  36273   90682500
December  54132  135330000


In [81]:
# quarterly report
print("\n=== Quarterly Sales Report ===")
df['Quarter'] = df['Date'].dt.to_period('Q')
quarterly_sales = df.groupby('Quarter')[['Unit', 'Sales']].sum()
print(quarterly_sales)


=== Quarterly Sales Report ===
           Unit      Sales
Quarter                   
2020Q4   136121  340302500
